In [1]:
appointments_df = spark.read.table("silver.appointments")
billing_df = spark.read.table("silver.billing")
doctors_df = spark.read.table("silver.doctors")
patients_df = spark.read.table("silver.patients")
treatments_df = spark.read.table("silver.treatments")

StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.functions import concat_ws
from pyspark.sql.functions import date_format
from pyspark.sql.functions import year, month

StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 4, Finished, Available, Finished, False)

In [3]:
dim_patient = patients_df.select(
    "patient_id",
    "first_name",
    "last_name",
    "gender",
    "date_of_birth",
    "registration_date",
    "insurance_provider"
)

dim_patient = dim_patient.withColumn(
    "patient_full_name",
    concat_ws(" ", dim_patient.first_name, dim_patient.last_name)
)

dim_patient.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold.dim_patient")

display(dim_patient)

StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ee667c78-5643-4bc3-b219-e7bbdc0d25fd)

In [4]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 6, Finished, Available, Finished, False)

DataFrame[]

In [5]:
spark.sql("SHOW TABLES IN gold").show()

StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 7, Finished, Available, Finished, False)

+--------------------+--------------------+-----------+
|           namespace|           tableName|isTemporary|
+--------------------+--------------------+-----------+
|`Enterprise Healt...|            dim_date|      false|
|`Enterprise Healt...|          dim_doctor|      false|
|`Enterprise Healt...|         dim_patient|      false|
|`Enterprise Healt...|       dim_treatment|      false|
|`Enterprise Healt...|   fact_appointments|      false|
|`Enterprise Healt...|        fact_billing|      false|
|`Enterprise Healt...|gold_doctor_perfo...|      false|
|`Enterprise Healt...|gold_revenue_summary|      false|
+--------------------+--------------------+-----------+



In [6]:
from pyspark.sql.functions import concat_ws

dim_doctor = doctors_df.select(
    "doctor_id",
    "first_name",
    "last_name",
    "specialization",
    "years_experience",
    "hospital_branch",
    "email"
)

dim_doctor = dim_doctor.withColumn(
    "doctor_full_name",
    concat_ws(" ", dim_doctor.first_name, dim_doctor.last_name)
)

dim_doctor.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold.dim_doctor")


StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 8, Finished, Available, Finished, False)

In [7]:
dim_treatment = treatments_df.select(
    "treatment_id",
    "treatment_type",
    "description",
    "cost"
).dropDuplicates(["treatment_id"])

dim_treatment = treatments_df.select(
    "treatment_id",
    "treatment_type",
    "description",
    "cost"
)

dim_treatment.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold.dim_treatment")

display(dim_treatment)

StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1e07c27a-900f-4da2-b300-8471451fb977)

In [8]:
spark.sql("SHOW TABLES IN gold").show()

StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 10, Finished, Available, Finished, False)

+--------------------+--------------------+-----------+
|           namespace|           tableName|isTemporary|
+--------------------+--------------------+-----------+
|`Enterprise Healt...|            dim_date|      false|
|`Enterprise Healt...|          dim_doctor|      false|
|`Enterprise Healt...|         dim_patient|      false|
|`Enterprise Healt...|       dim_treatment|      false|
|`Enterprise Healt...|   fact_appointments|      false|
|`Enterprise Healt...|        fact_billing|      false|
|`Enterprise Healt...|gold_doctor_perfo...|      false|
|`Enterprise Healt...|gold_revenue_summary|      false|
+--------------------+--------------------+-----------+



In [9]:
from pyspark.sql.functions import when, lit

fact_appointments = appointments_df.select(
    "appointment_id",
    "patient_id",
    "doctor_id",
    "appointment_date",
    "appointment_time",
    "reason_for_visit",
    "status"
)

fact_appointments = fact_appointments \
    .withColumn("appointment_count", lit(1)) \
    .withColumn(
        "is_completed",
        when(fact_appointments.status == "Completed", 1).otherwise(0)
    ) \
    .withColumn(
        "is_cancelled",
        when(fact_appointments.status == "Cancelled", 1).otherwise(0)
    ) \
    .withColumn(
        "is_no_show",
        when(fact_appointments.status == "No-show", 1).otherwise(0)
    )

StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 11, Finished, Available, Finished, False)

In [10]:
fact_billing = billing_df.select(
    "bill_id",
    "patient_id",
    "treatment_id",
    "bill_date",
    "amount",
    "payment_method",
    "payment_status"
)


fact_billing = fact_billing \
    .withColumn("bill_count", lit(1)) \
    .withColumn(
        "is_paid",
        when(fact_billing.payment_status == "Paid", 1).otherwise(0)
    )

fact_billing = fact_billing \
    .withColumn("bill_count", lit(1)) \
    .withColumn(
        "is_paid",
        when(fact_billing.payment_status == "Paid", 1).otherwise(0)
    )

fact_billing.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold.fact_billing")

display(fact_billing)



fact_appointments.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold.fact_appointments")


StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3ea8629b-e09c-4df0-ad4c-ddd903b5ed86)

In [11]:
spark.sql("SHOW TABLES IN gold").show()

StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 13, Finished, Available, Finished, False)

+--------------------+--------------------+-----------+
|           namespace|           tableName|isTemporary|
+--------------------+--------------------+-----------+
|`Enterprise Healt...|            dim_date|      false|
|`Enterprise Healt...|          dim_doctor|      false|
|`Enterprise Healt...|         dim_patient|      false|
|`Enterprise Healt...|       dim_treatment|      false|
|`Enterprise Healt...|   fact_appointments|      false|
|`Enterprise Healt...|        fact_billing|      false|
|`Enterprise Healt...|gold_doctor_perfo...|      false|
|`Enterprise Healt...|gold_revenue_summary|      false|
+--------------------+--------------------+-----------+



In [12]:
from pyspark.sql.functions import sum, col

gold_doctor_performance = fact_appointments.groupBy(
    "doctor_id"
).agg(
    sum("appointment_count").alias("total_appointments"),
    sum("is_completed").alias("completed_appointments"),
    sum("is_cancelled").alias("cancelled_appointments"),
    sum("is_no_show").alias("no_show_appointments")
)

gold_doctor_performance = gold_doctor_performance \
    .withColumn(
        "cancellation_rate",
        col("cancelled_appointments") / col("total_appointments")
    ) \
    .withColumn(
        "no_show_rate",
        col("no_show_appointments") / col("total_appointments")
    )

gold_doctor_performance = gold_doctor_performance.join(
    dim_doctor.select(
        "doctor_id",
        "doctor_full_name",
        "specialization",
        "hospital_branch",
        "years_experience"
    ),
    on="doctor_id",
    how="left"
)


gold_doctor_performance.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold.gold_doctor_performance")


StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 14, Finished, Available, Finished, False)

In [13]:
from pyspark.sql.functions import avg

gold_revenue_summary = fact_billing.groupBy(
    "payment_method",
    "payment_status"
).agg(
    sum("amount").alias("total_revenue"),
    sum("bill_count").alias("total_bills"),
    avg("amount").alias("avg_bill_amount")
)

gold_revenue_summary.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold.gold_revenue_summary")

StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 15, Finished, Available, Finished, False)

In [14]:
from pyspark.sql.functions import *

dim_date = (
    spark.range(0, 365)
    .select(
        date_add(lit("2023-01-01").cast("date"), col("id").cast("int")).alias("date")
    )
)

dim_date = dim_date \
    .withColumn("year", year("date")) \
    .withColumn("quarter", quarter("date")) \
    .withColumn("month_number", month("date")) \
    .withColumn("month_name", date_format("date", "MMMM")) \
    .withColumn("month_short", date_format("date", "MMM")) \
    .withColumn("year_month", date_format("date", "yyyy-MM")) \
    .withColumn("day", dayofmonth("date"))

dim_date.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold.dim_date")

StatementMeta(, 18fd0a6b-7911-41ab-8ffb-07afa0e0c5e1, 16, Finished, Available, Finished, False)